
# DS2002 — Homework: Stocks ETL
## APIs + DataFrames + Databases

In this homework you will build a small end-to-end ETL pipeline:

1. **Extract** data from multiple API endpoints (more than one endpoint is required)
2. **Transform** the raw JSON into clean Pandas DataFrames
3. **Cull** (remove) columns you do not need
4. **Transform** the cleaned data (compute new features)
5. **Load** the resulting DataFrames into a SQLite database
6. **Prove** the load worked by running SQL queries against your new tables

You will use the finance API described at `financeapi.net` (base URL: `https://yfapi.net`).

This assignment is designed to feel like real systems work:
- APIs return nested JSON
- you must decide what columns matter
- you must build tables that are useful later



## Submission

Submit the **Kaggle Notebook URL**.

Your notebook must show:
- completed TODO cells
- visible outputs for key steps (DataFrames displayed)
- SQL proof queries at the end (outputs visible)



## Grading Rubric (100 points)

### A) Extract: API Calls (35 points)
You must successfully call **at least 3 endpoints**.

- 15 pts: `/v6/finance/quote` for multiple tickers
- 10 pts: `/v8/finance/chart/{ticker}` for historical prices (time series)
- 10 pts: One additional endpoint of your choice (examples below)

Acceptable “third endpoint” choices include (pick ONE):
- `/v6/finance/autocomplete`
- `/v1/finance/trending/{region}`
- `/v6/finance/quote/marketSummary`
- `/v11/finance/quoteSummary/{symbol}` (requires `modules`)

### B) Transform in Pandas (35 points)
- 10 pts: Cull columns (keep only useful fields; show the before/after column lists)
- 15 pts: At least **3 real transformations** that change meaning (not just renaming)
- 10 pts: At least **1 join/merge** between DataFrames (ex: quote table + latest close)

### C) Load to SQLite (20 points)
- 10 pts: Load at least **3 DataFrames** into SQL using `.to_sql(...)`
- 10 pts: Tables have sensible names and useful columns

### D) Proof Queries (10 points)
- 10 pts: Run at least **3 SQL queries** that prove the data is present and makes sense

### Reproducibility (penalty up to -10 points)
If the notebook does not run top-to-bottom without manual fixes, points may be deducted.



## Required Setup: API Key (Kaggle)

This API requires a key.

In Kaggle:
1. Turn **Internet ON** in notebook settings.
2. Add a Kaggle **Secret** named: `YHFINANCE_API_KEY`
3. Restart the session and run the setup cell below.

Do not paste keys into notebooks after you turn them in...make sure I can see your results and then delete the key. Treat keys like passwords.



## Part 0 — Setup (Run This)

This cell:
- imports libraries
- creates a SQLite connection
- defines `q(...)` for SQL proof queries
- defines `get_json(...)` for consistent API requests

If your key is missing, this cell will tell you what to do.


In [153]:

import os
import time
import sqlite3
import requests
import pandas as pd

# SQLite database (in-memory is fine for homework)
conn = sqlite3.connect(":memory:")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

BASE_URL = "https://yfapi.net"

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
API_KEY = user_secrets.get_secret("yfinance")


if not API_KEY:
    print("No API key found.")
    print("Kaggle → Add-ons/Settings → Secrets → add YHFINANCE_API_KEY")
    print("Then restart the session and re-run this cell.")
else:
    print("API key loaded (not printed).")

def get_json(endpoint: str, params: dict | None = None, pause_s: float = 0.25) -> dict:
    """Call the Finance API endpoint and return parsed JSON."""
    if not API_KEY:
        raise RuntimeError("Missing API key. Set YHFINANCE_API_KEY as a Kaggle Secret.")

    url = BASE_URL + endpoint
    headers = {
        "accept": "application/json",
        "X-API-KEY": API_KEY
    }
    resp = requests.get(url, headers=headers, params=params, timeout=30)
    time.sleep(pause_s)  # helps avoid rate limiting

    if resp.status_code != 200:
        raise RuntimeError(
            f"Request failed ({resp.status_code})\n"
            f"URL: {resp.url}\n"
            f"Body (first 500 chars): {resp.text[:500]}"
        )
    return resp.json()

print("Setup ready.")


API key loaded (not printed).
Setup ready.



## Part 1 — Choose Your Tickers (TODO)

Pick **3–6** stock tickers.

Guidance:
- choose large, well-known companies
- avoid very new or obscure tickers

Example set (use your own if you want): `AAPL, MSFT, AMZN, TSLA, NVDA`


In [154]:

# TODO: choose 3–6 tickers
tickers = ["AAPL", "MSFT", "TSLA"]

tickers


['AAPL', 'MSFT', 'TSLA']


## Part 2 — Extract #1: Quote Data for Multiple Tickers (TODO)

Endpoint: `GET /v6/finance/quote`

This endpoint can take multiple symbols at once using a comma-separated list.

Your job:
1. Build the comma-separated symbol list
2. Call the endpoint
3. Convert the results into a DataFrame `df_quote`
4. Display `df_quote.head()`

Hint:
- results are commonly nested under: `quoteResponse -> result`


In [155]:

# TODO
symbols = ",".join(tickers)
raw_quote = get_json("/v6/finance/quote", params={"symbols": symbols, "region": "US", "lang": "en"})
data = raw_quote["quoteResponse"]["result"]
df_quote = pd.DataFrame(data)
df_quote.head()

,language,region,quoteType,typeDisp,quoteSourceName,triggerable,customPriceAlertConfidence,currency,marketState,corporateActions,...,marketCap,forwardPE,priceToBook,sourceInterval,exchangeDataDelayedBy,averageAnalystRating,tradeable,cryptoTradeable,displayName,symbol
0,en-US,US,EQUITY,Equity,Nasdaq Real Time Price,True,HIGH,USD,POSTPOST,[],...,3885396656128,28.474302,44.073025,15,0,1.9 - Buy,False,False,Apple,AAPL
1,en-US,US,EQUITY,Equity,Nasdaq Real Time Price,True,HIGH,USD,POSTPOST,[],...,2969978273792,21.199400,7.594792,15,0,1.3 - Strong Buy,False,False,Microsoft,MSFT
2,en-US,US,EQUITY,Equity,Nasdaq Real Time Price,True,HIGH,USD,POSTPOST,[],...,1543450263552,146.673700,18.784310,15,0,2.5 - Hold,False,False,Tesla,TSLA



## Part 3 — Cull Columns (TODO)

API responses contain many fields.
Your job is to keep only what your pipeline needs.

Minimum columns to keep (you may keep more if you justify them):
- `symbol`
- `shortName` (or `longName`)
- `regularMarketPrice`
- `regularMarketChange`
- `regularMarketChangePercent`
- `regularMarketVolume`
- `marketCap`
- `currency`

Your deliverable:
1. Print the original column count and column list
2. Create `df_quote_clean`
3. Print the cleaned column count and column list
4. Display `df_quote_clean.head()`


In [156]:

# TODO
print(len(df_quote.columns), "columns in df")
print("------------------------------------")
print(df_quote.columns, "\n")

columns_to_keep = ["symbol", "shortName", "regularMarketPrice", "regularMarketChange", "regularMarketChangePercent"
                  , "regularMarketVolume", "marketCap", "currency"]

df_quote_clean = df_quote[columns_to_keep].copy()

print(len(df_quote_clean.columns), "columns in df_cleaned")
print("------------------------------------")
print(df_quote_clean.columns)

df_quote_clean.head()


86 columns in df
------------------------------------
Index(['language', 'region', 'quoteType', 'typeDisp', 'quoteSourceName',
       'triggerable', 'customPriceAlertConfidence', 'currency', 'marketState',
       'corporateActions', 'postMarketTime', 'regularMarketTime', 'shortName',
       'longName', 'exchange', 'messageBoardId', 'exchangeTimezoneName',
       'exchangeTimezoneShortName', 'gmtOffSetMilliseconds', 'market',
       'esgPopulated', 'regularMarketChangePercent', 'regularMarketPrice',
       'hasPrePostMarketData', 'firstTradeDateMilliseconds', 'priceHint',
       'postMarketChangePercent', 'postMarketPrice', 'postMarketChange',
       'regularMarketChange', 'regularMarketDayHigh', 'regularMarketDayRange',
       'regularMarketDayLow', 'regularMarketVolume',
       'regularMarketPreviousClose', 'bid', 'ask', 'bidSize', 'askSize',
       'fullExchangeName', 'financialCurrency', 'regularMarketOpen',
       'averageDailyVolume3Month', 'averageDailyVolume10Day',
       'fifty

,symbol,shortName,regularMarketPrice,regularMarketChange,regularMarketChangePercent,regularMarketVolume,marketCap,currency
0,AAPL,Apple Inc.,264.35,0.470001,0.178112,33598417,3885396656128,USD
1,MSFT,Microsoft Corporation,399.60,2.740020,0.690425,23013887,2969978273792,USD
2,TSLA,"Tesla, Inc.",411.32,0.690002,0.168035,45533478,1543450263552,USD



## Part 4 — Transform #1: Make the Quote Table More Useful (TODO)

Add these columns to `df_quote_clean`:

1. `marketCap_B` = marketCap expressed in **billions**
2. `price_rounded` = regularMarketPrice rounded to 2 decimals
3. `change_pct_rounded` = regularMarketChangePercent rounded to 2 decimals

Then:
- sort by `marketCap_B` descending
- display the sorted DataFrame

This is a common “transform” step in ETL: making fields more usable for humans and downstream analysis.


In [157]:

# TODO
df_quote_clean["marketCap_B"] = df_quote_clean["marketCap"] / 1_000_000_000
df_quote_clean["price_rounded"] = round(df_quote_clean["regularMarketPrice"], 2)
df_quote_clean["change_pct_rounded"] = round(df_quote_clean["regularMarketChangePercent"], 2)
df_quote_clean = df_quote_clean.sort_values("marketCap_B", ascending=False)
df_quote_clean.head()

,symbol,shortName,regularMarketPrice,regularMarketChange,regularMarketChangePercent,regularMarketVolume,marketCap,currency,marketCap_B,price_rounded,change_pct_rounded
0,AAPL,Apple Inc.,264.35,0.470001,0.178112,33598417,3885396656128,USD,3885.396656,264.35,0.18
1,MSFT,Microsoft Corporation,399.60,2.740020,0.690425,23013887,2969978273792,USD,2969.978274,399.60,0.69
2,TSLA,"Tesla, Inc.",411.32,0.690002,0.168035,45533478,1543450263552,USD,1543.450264,411.32,0.17



## Part 5 — Extract #2: Historical Prices for One Ticker (TODO)

Endpoint: `GET /v8/finance/chart/{ticker}`

You will choose ONE ticker from your list as the focus ticker.
You will build a daily price table from the chart endpoint.

Your deliverable:
1. Set `FOCUS_TICKER`
2. Call the chart endpoint for a range like `"1mo"` or `"3mo"` and interval `"1d"`
3. Build `df_prices` with columns:
   - `symbol`
   - `date` (as datetime)
   - `close`
4. Drop rows where close is missing
5. Display `df_prices.head()` and `df_prices.tail()`

Hints:
- timestamps are often in seconds (use `pd.to_datetime(..., unit="s")`)
- close values are usually nested under indicators -> quote -> close


In [158]:

# TODO
FOCUS_TICKER = "AAPL"
raw_chart = get_json(f"/v8/finance/chart/{FOCUS_TICKER}", params={"range": "1mo", "interval": "1d", "region": "US", "lang": "en"})

result = raw_chart["chart"]["result"][0]

timestamps = result["timestamp"]
closes = result["indicators"]["quote"][0]["close"]

df_prices = pd.DataFrame({"symbol": FOCUS_TICKER, "date": pd.to_datetime(timestamps, unit="s"), "close": closes})

df_prices = df_prices.dropna(subset=["close"])
df_prices.head()


,symbol,date,close
0,AAPL,2026-01-20 14:30:00,246.699997
1,AAPL,2026-01-21 14:30:00,247.649994
2,AAPL,2026-01-22 14:30:00,248.350006
3,AAPL,2026-01-23 14:30:00,248.039993
4,AAPL,2026-01-26 14:30:00,255.410004



## Part 6 — Transform #2: Compute Returns and Rolling Metrics (TODO)

Time series are valuable because we can compute signals.

Add these columns to `df_prices`:
1. `daily_return` = percent change of close from the previous day
2. `rolling_7d_avg_close` = 7-day rolling mean of close
3. `rolling_7d_avg_return` = 7-day rolling mean of daily_return

Deliverable:
- show the last 10 rows

Hints:
- sort by date first
- use `pct_change()`
- use `rolling(7).mean()`


In [159]:

# TODO
df_prices = df_prices.sort_values(by='date')
df_prices["daily_return"] = df_prices["close"].pct_change()
df_prices["rolling_7d_avg_close"] = df_prices["close"].rolling(7).mean()
df_prices["rolling_7d_avg_return"] = df_prices["daily_return"].rolling(7).mean()

df_prices.tail(10)


,symbol,date,close,daily_return,rolling_7d_avg_close,rolling_7d_avg_return
11,AAPL,2026-02-04 14:30:00,276.489990,0.026013,264.064287,0.011509
12,AAPL,2026-02-05 14:30:00,275.910004,-0.002098,266.584290,0.009610
13,AAPL,2026-02-06 14:30:00,278.119995,0.008010,269.681431,0.011766
14,AAPL,2026-02-09 14:30:00,274.619995,-0.012584,272.015717,0.008944
15,AAPL,2026-02-10 14:30:00,273.679993,-0.003423,274.044285,0.007791
16,AAPL,2026-02-11 14:30:00,275.500000,0.006650,274.828570,0.002944
17,AAPL,2026-02-12 14:30:00,261.730011,-0.049982,273.721427,-0.003916
18,AAPL,2026-02-13 14:30:00,255.779999,-0.022733,270.762857,-0.010880
19,AAPL,2026-02-17 14:30:00,263.880005,0.031668,269.044285,-0.006056
20,AAPL,2026-02-18 14:30:00,264.350006,0.001781,267.077144,-0.006946



## Part 7 — Extract #3: A Third Endpoint (Pick One) (TODO)

You must call one additional endpoint beyond quote + chart.

Pick ONE option below and build a DataFrame called `df_extra`:

Option A: Autocomplete
- endpoint: `/v6/finance/autocomplete`
- params might include: `query`, `region`

Option B: Trending
- endpoint: `/v1/finance/trending/{region}`
- example region: `US`

Option C: Market Summary
- endpoint: `/v6/finance/quote/marketSummary`
- params may include: `region` and/or `lang`

Option D: Quote Summary
- endpoint: `/v11/finance/quoteSummary/{symbol}`
- requires a `modules` parameter (request only a small set)

Deliverable:
1. Make the call
2. Convert to DataFrame `df_extra`
3. Cull columns to a small meaningful set
4. Display `df_extra.head()`


In [160]:

# TODO
raw_chart = get_json("/v11/finance/quoteSummary/AAPL", params={
    "modules": "earningsHistory"
})
data = raw_chart['quoteSummary']['result'][0]
eh = data['earningsHistory']['history']
rows = []
for quarter in eh:
   rows.append({
       "epsActual": quarter["epsActual"]["raw"],
       "epsEstimate": quarter["epsEstimate"]["raw"],
       "epsDifference": quarter["epsDifference"]["raw"],
       "surprisePercent": quarter["surprisePercent"]["raw"],
       "quarter": quarter["quarter"]["raw"],
       "period": quarter["period"]
   })
df_extra = pd.DataFrame(rows)
df_extra['quarter'] = pd.to_datetime(df_extra['quarter'], unit='s')
df_extra['period'] = df_extra['period'].str.replace('-', '')
df_extra.head()


,epsActual,epsEstimate,epsDifference,surprisePercent,quarter,period
0,1.65,1.62253,0.03,0.0169,2025-03-31,4q
1,1.57,1.42572,0.14,0.1012,2025-06-30,3q
2,1.85,1.76993,0.08,0.0452,2025-09-30,2q
3,2.84,2.67080,0.17,0.0634,2025-12-31,1q



## Part 8 — Transform #3: Join DataFrames (TODO)

ETL pipelines often combine data sources.

Goal:
- Take the latest close from `df_prices`
- Merge it into `df_quote_clean` on `symbol`
- Compute a new column: `price_minus_latest_close`

Deliverable:
1. Create `df_latest` (1 row) with: symbol, latest_date, latest_close
2. Merge into a new DataFrame `df_quote_enriched`
3. Add `price_minus_latest_close`
4. Display `df_quote_enriched`


In [161]:

# TODO
df_latest = df_prices.tail(1)[["symbol", "date", "close"]].rename(columns={
    "date": "latest_date",
    "close": "latest_close"
})
df_quote_enriched = pd.merge(df_quote_clean, df_latest, on="symbol", how="left")
df_quote_enriched['price_minus_latest_close'] = df_quote_enriched['price_rounded']-df_quote_enriched['latest_close']
df_quote_enriched


,symbol,shortName,regularMarketPrice,regularMarketChange,regularMarketChangePercent,regularMarketVolume,marketCap,currency,marketCap_B,price_rounded,change_pct_rounded,latest_date,latest_close,price_minus_latest_close
0,AAPL,Apple Inc.,264.35,0.470001,0.178112,33598417,3885396656128,USD,3885.396656,264.35,0.18,2026-02-18 14:30:00,264.350006,-0.000006
1,MSFT,Microsoft Corporation,399.60,2.740020,0.690425,23013887,2969978273792,USD,2969.978274,399.60,0.69,NaT,NaN,NaN
2,TSLA,"Tesla, Inc.",411.32,0.690002,0.168035,45533478,1543450263552,USD,1543.450264,411.32,0.17,NaT,NaN,NaN



## Part 9 — Load: Write DataFrames Into SQLite (TODO)

You must load at least 3 tables:
- `quotes`
- `prices`
- `extra`

Use:
- `to_sql(..., if_exists="replace", index=False)`

Deliverable:
- list the SQLite tables after writing


In [162]:

# TODO
df_quote_enriched.to_sql("quotes", conn, if_exists="replace", index=False)
df_prices.to_sql("prices", conn, if_exists="replace", index=False)
df_extra.to_sql("extra", conn, if_exists="replace", index=False)

q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")


,name
0,extra
1,prices
2,quotes



## Part 10 — Proof: SQL Queries (TODO)

Run at least 3 SQL queries to prove your load worked.
Use the `q(...)` helper.

Good proof queries include:
- row counts
- sample rows
- simple aggregates / sanity checks

Deliverable:
- Fill in 3 queries below and show outputs.


In [163]:

# TODO Query 1: Row counts for each table
#
q("""
    SELECT 'quotes' AS table_name, COUNT(*) AS row_count FROM quotes
    UNION ALL
    SELECT 'prices', COUNT(*) FROM prices
    UNION ALL
    SELECT 'extra', COUNT(*) FROM extra
""")




,table_name,row_count
0,quotes,3
1,prices,21
2,extra,4


In [164]:

# TODO Query 2: Show the largest market cap stock in your quotes table
q("SELECT * FROM quotes ORDER BY marketcap DESC LIMIT 1")


,symbol,shortName,regularMarketPrice,regularMarketChange,regularMarketChangePercent,regularMarketVolume,marketCap,currency,marketCap_B,price_rounded,change_pct_rounded,latest_date,latest_close,price_minus_latest_close
0,AAPL,Apple Inc.,264.35,0.470001,0.178112,33598417,3885396656128,USD,3885.396656,264.35,0.18,2026-02-18 14:30:00,264.350006,-0.000006


In [165]:

# TODO Query 3: Show the most recent 5 price rows for your focus ticker
q("SELECT * FROM prices ORDER BY date DESC LIMIT 5")




,symbol,date,close,daily_return,rolling_7d_avg_close,rolling_7d_avg_return
0,AAPL,2026-02-18 14:30:00,264.350006,0.001781,267.077144,-0.006946
1,AAPL,2026-02-17 14:30:00,263.880005,0.031668,269.044285,-0.006056
2,AAPL,2026-02-13 14:30:00,255.779999,-0.022733,270.762857,-0.010880
3,AAPL,2026-02-12 14:30:00,261.730011,-0.049982,273.721427,-0.003916
4,AAPL,2026-02-11 14:30:00,275.500000,0.006650,274.828570,0.002944



## Part 11 — Reflection (Required)

Write 3–6 sentences answering:
1. Which endpoint was easiest to work with, and why?
2. What was hardest part of this ETL pipeline?
3. If this ran daily in production, what would you automate or change?


In [166]:

reflection = """
1. /v6/finance/quote gave me the least trouble because i thought that the data was more straightforward compared to the other endpoints
2. I think the nested JSON was difficult for me just becuase there are so many brackets it was hard for me to keep track of what's what
3. I would automate the json nesting. that was kind of a pain to do for each endpoint.
"""

print(reflection)



1. /v6/finance/quote gave me the least trouble because i thought that the data was more straightforward compared to the other endpoints
2. I think the nested JSON was difficult for me just becuase there are so many brackets it was hard for me to keep track of what's what
3. I would automate the json nesting. that was kind of a pain to do for each endpoint.




## Done

Before submitting:
- Run the notebook top-to-bottom (Restart & Run All)
- Confirm your proof queries show real rows
- Submit the Kaggle Notebook URL in Canvas
